In [1]:
from pathlib import Path

import airbench
import torch
import torch.nn.functional as F
from airbench.utils import CIFAR_MEAN, CIFAR_STD
from tqdm.autonotebook import tqdm
from visualization import save_image_grid_in_background

from model import CifarVaeModel

ARTIFACTS_DIR = (Path.cwd() / "artifacts" / "vae").resolve()
SAMPLES_DIR = ARTIFACTS_DIR / "generated_samples"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

/tmp/ipykernel_4138016/447326001.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


### Define and load data

In [2]:
DATA_DIR = Path.home() / ".cache" / "airbench-cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_loader = airbench.CifarLoader(
    str(DATA_DIR),
    train=True,
    batch_size=1024,
    aug=dict(flip=True, translate=4, cutout=16),
)
test_loader = airbench.CifarLoader(
    str(DATA_DIR),
    train=False,
    batch_size=1000,
)

# VAE serves as a tokenizer for downstream generative modeling — train on all 60k.
train_loader.images = torch.cat([train_loader.images, test_loader.images])
train_loader.labels = torch.cat([train_loader.labels, test_loader.labels])
print(f"train: {len(train_loader.images)} images, {len(train_loader)} batches of {train_loader.batch_size}")

train: 60000 images, 58 batches of 1024


### Define model

In [ ]:
device = "cuda:1"
model = CifarVaeModel(
    stage_channels=(64, 128, 256, 512),
    latent_channels=4,
    blocks_per_stage=2,
).to(device)

with torch.no_grad():
    probe_mu, _ = model.encoder(torch.zeros(1, 3, 32, 32, device=device))
latent_shape = tuple(probe_mu.shape[1:])
print(f"input:  (3, 32, 32) = {3 * 32 * 32} dims/image")
print(f"latent: {latent_shape} = {probe_mu[0].numel()} dims/image")
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

model = torch.compile(model)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

input:  (3, 32, 32) = 3072 dims/image
latent: (4, 4, 4) = 64 dims/image
params: 28.59M


## VAE training loop

In [ ]:
beta = 1.0
epochs = 50

# Same z every epoch so the saved grids show how the *same* prior samples evolve.
fixed_z = torch.randn(64, *latent_shape, device=device)
sample_futures: list = []

model.train()
for epoch in range(epochs):
    pbar = tqdm(train_loader, desc=f"epoch {epoch}", leave=True)
    for images, _ in pbar:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            x_hat, mu, logvar = model(images)
            recon = F.mse_loss(x_hat, images, reduction="none").sum(dim=(1, 2, 3)).mean()
            kl = 0.5 * (mu.pow(2) + logvar.exp() - 1 - logvar).sum(dim=(1, 2, 3)).mean()
            loss = recon + beta * kl
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        pbar.set_postfix(recon=f"{recon.item():.2f}", kl=f"{kl.item():.2f}")

    model.eval()
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        samples = model.decoder(fixed_z).float()
    sample_futures.append(
        save_image_grid_in_background(
            images=samples,
            path=SAMPLES_DIR / f"epoch_{epoch:03d}.png",
            ncols=8,
            title=f"epoch {epoch} — prior samples",
        )
    )
    model.train()

# Surface any background-render failures before moving on.
for fut in sample_futures:
    fut.result()

epoch 0:   0%|          | 0/58 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/58 [00:00<?, ?it/s]

### Post-training

In [ ]:
# After training, save to /home/nlyu/Code/diffusive-latent-generation/experiments/cifar-smoke VAE / encoder.pkl, decoder.pkl
# Also save the image artifacts (train + val)

### CIFAR stats computation

In [ ]:
import time

import torch.nn.functional as F
from precision_recall import PrecisionRecallConfig
from torchvision.models import Inception_V3_Weights, inception_v3

# Inception-V3 pool3 features (2048-d) are the de facto standard for improved P&R.
# airbench gives us images already normalised by CIFAR mean/std; Inception expects
# 299x299 inputs normalised by ImageNet mean/std, so we undo CIFAR norm, resize,
# then re-normalise for ImageNet.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
_cifar_mean = CIFAR_MEAN.to(device).view(1, 3, 1, 1)
_cifar_std = CIFAR_STD.to(device).view(1, 3, 1, 1)

inception = inception_v3(
    weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True
).to(device).eval()
inception.fc = torch.nn.Identity()  # expose 2048-d pool3 features


@torch.inference_mode()
def inception_features(images_cifar_norm, *, batch=256):
    """Map airbench-normalised CIFAR-shaped (B, 3, 32, 32) images to (B, 2048) features."""
    n = images_cifar_norm.shape[0]
    out = torch.empty(n, 2048, device=device, dtype=torch.float32)
    for i in range(0, n, batch):
        x = images_cifar_norm[i : i + batch].to(device).float()
        x = x * _cifar_std + _cifar_mean
        x = F.interpolate(x, size=299, mode="bilinear", align_corners=False)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            f = inception(x).float()
        out[i : i + f.shape[0]] = f
    return out


pr_cfg = PrecisionRecallConfig(nearest_k=3, distance_chunk=4096)

In [ ]:
# (1) CIFAR vs CIFAR self-baseline: split the 60k airbench images into two random
# halves and compute improved P&R between them. This is the empirical ceiling for
# any generator evaluated with k=3 NN on Inception-V3 pool3 features.
t0 = time.perf_counter()
real_features = inception_features(train_loader.images)
torch.cuda.synchronize()
print(f"[extract real features {real_features.shape[0]}] "
      f"{time.perf_counter() - t0:5.1f}s -> {tuple(real_features.shape)}")

perm_r = torch.randperm(real_features.shape[0], device=device)
half_r = real_features.shape[0] // 2
real_A = real_features[perm_r[:half_r]]
real_B = real_features[perm_r[half_r:2 * half_r]]

torch.cuda.synchronize(); t0 = time.perf_counter()
real_self = pr_cfg.evaluate(data=real_A, generated=real_B)
torch.cuda.synchronize()
print(f"[CIFAR self {half_r} vs {half_r}] {time.perf_counter() - t0:5.2f}s  {real_self}")

### Generated-sample stats computation

In [ ]:
# (2) VAE samples vs themselves: draw N latents from the prior N(0, I), decode,
# then split the resulting images half/half and compute self-P&R. This measures
# the spread of the generator's own output in feature space -- an internal
# consistency check independent of fidelity to CIFAR.
n_samples = real_features.shape[0]  # match the real-side count
sample_batch = 512


@torch.inference_mode()
def sample_vae(n, *, batch):
    out = torch.empty(n, 3, 32, 32, device=device, dtype=torch.float32)
    for i in range(0, n, batch):
        b = min(batch, n - i)
        z = torch.randn(b, *latent_shape, device=device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            x = model.decoder(z).float()
        out[i : i + b] = x
    return out


model.eval()
t0 = time.perf_counter()
gen_images = sample_vae(n_samples, batch=sample_batch)
torch.cuda.synchronize()
print(f"[sample VAE {n_samples}] {time.perf_counter() - t0:5.1f}s -> {tuple(gen_images.shape)}")

t0 = time.perf_counter()
gen_features = inception_features(gen_images)
torch.cuda.synchronize()
print(f"[extract gen features] {time.perf_counter() - t0:5.1f}s")

perm_g = torch.randperm(gen_features.shape[0], device=device)
half_g = gen_features.shape[0] // 2
gen_A = gen_features[perm_g[:half_g]]
gen_B = gen_features[perm_g[half_g:2 * half_g]]

torch.cuda.synchronize(); t0 = time.perf_counter()
gen_self = pr_cfg.evaluate(data=gen_A, generated=gen_B)
torch.cuda.synchronize()
print(f"[VAE   self {half_g} vs {half_g}] {time.perf_counter() - t0:5.2f}s  {gen_self}")

# For reference, also report cross P&R: how well the generator's manifold matches CIFAR.
torch.cuda.synchronize(); t0 = time.perf_counter()
cross = pr_cfg.evaluate(data=real_features, generated=gen_features)
torch.cuda.synchronize()
print(f"[CIFAR vs VAE  cross]                {time.perf_counter() - t0:5.2f}s  {cross}")